# Task 2 and Task 3 Analysis Evidence - xfan0282


## Member Scope

This notebook is the individual Task 2 and Task 3 evidence for `xfan0282`.


In [39]:
from dataclasses import replace
import pandas as pd

from data2001.config import load_settings
from data2001.db.engine import create_engine_from_settings
from data2001.pipeline import execute_workflow_steps

# Plotly chart builders
from data2001.task4.charts import (
    build_bottom_sa2_bar,
    build_poi_group_distribution,
    build_score_histogram,
    build_score_income_scatter,
    build_top_sa2_bar,
)
from data2001.task4.tables import build_sa4_summary_table, build_top_bottom_table
from data2001.task4.maps import build_score_choropleth_map, build_poi_density_choropleth_map, build_poi_point_scatter_map

from data2001.task4.queries import (
    load_api_extraction_summary,
    load_correlation_summary,
    load_poi_group_counts,
    load_poi_points,
    load_sa2_scores,
    load_score_income,
    load_score_input_summary,
    load_spatial_join_summary,
)


MEMBER_UNIKEY = "xfan0282"

base_settings = load_settings("configs/local.yaml") # load group config and local database/API paths
member_sa4 = (base_settings.task2_import.selected_sa4_by_member.get(MEMBER_UNIKEY) or "").strip() # read xfan0282's SA4
if not member_sa4:
    raise ValueError(f"No SA4 configured for {MEMBER_UNIKEY}. Fill configs/local.yaml selected_sa4_by_member.")

settings = replace(
    base_settings, # start from normal local.yaml settings
    task2_import=replace(
        base_settings.task2_import,
        crawl_scope="selected_sa4", # restrict Task 2 imports to selected SA4 mode
        selected_sa4_by_member={MEMBER_UNIKEY: member_sa4}, # keep only xfan0282's SA4 for this notebook run
    ),
    task3_score=replace(base_settings.task3_score, score_universe="selected_sa4"), # score relative to this SA4 only
)
engine = create_engine_from_settings(settings.database) # SQLAlchemy engine for PostgreSQL/PostGIS queries

member_scope = pd.DataFrame([{"unikey": MEMBER_UNIKEY, "selected_sa4": member_sa4}])
display(member_scope) # confirm the member-specific SA4 scope before running workflow

,unikey,selected_sa4
0,xfan0282,Sydney - City and Inner South


## Single-SA4 Full Workflow Run

This cell rebuilds the personal Task 2/3 data for `xfan0282` from a clean local database.


In [40]:
workflow_steps = [
    # only run init_db once to create PostgreSQL/PostGIS schema, tables, and indexes
    "init_db",
    # for a full rebuild, clear_db can be rerun to remove old rows without dropping tables/indexes
    "clear_db",
    # downloads SA4/SA2 polygons and population from ArcGIS APIs
    "import_boundaries",
    # checks the geometry relationship using PostGIS queries between SA2 and SA4 to confirm correct membership
    "validate_boundaries",
    # loops through each SA2 bbox, downloads NSW POIs, cleans them, and assigns them with PostGIS
    "import_poi",
    # loads median income for interpretation
    "import_income",
    # calculates z-score/sigmoid scores and score-income correlation
    "compute_score",
]

workflow_summary = execute_workflow_steps(
    engine, # PostgreSQL/PostGIS connection for workflow steps to run SQL queries
    settings, # local.yaml config with member-specific overrides for this notebook
    workflow_steps, # ordered workflow step ids defined in pipeline.py
    title=f"{MEMBER_UNIKEY} single-SA4 full rebuild", # title used only for workflow logging
)
display(workflow_summary) # display workflow row counts and timing evidence

{'init_db': 'done',
 'clear_db': 'done',
 'sa4': 1,
 'sa2': 27,
 'population': 2473,
 'boundary_selected_sa4': 1,
 'boundary_sa2_checked': 27,
 'boundary_sa2_valid': 27,
 'boundary_sa2_invalid': 0,
 'boundary_min_coverage_ratio': 0.9999999999999983,
 'boundary_coverage_threshold': 0.999,
 'boundary_point_failures': 0,
 'boundary_missing_parent_sa4': 0,
 'sa2_bbox_requests': 27,
 'raw_responses': 27,
 'raw_features_seen': 3396,
 'clean_features_seen': 2203,
 'fetch_seconds': 3.227801820044988,
 'persist_seconds': 0.04002071899594739,
 'clean_seconds': 0.1385727169981692,
 'load_seconds': 0.043135369996889494,
 'income': 2454,
 'scores': 25,
 'correlations': 2}

### Boundary Validation Summary

This table checks the imported SA2 polygons against the selected SA4.


In [41]:
boundary_validation_keys = [
    # confirms one selected SA4 was validated
    "boundary_selected_sa4",
    # number of SA2 polygons checked using PostGIS geometry queries
    "boundary_sa2_checked",
    # valid SA2 count after ST_Covers and area coverage checks
    "boundary_sa2_valid",
    # invalid SA2 count; this should be 0 for a clean selected-SA4 run
    "boundary_sa2_invalid",
    # minimum SA2/SA4 coverage ratio found across imported SA2 polygons
    "boundary_min_coverage_ratio",
    # threshold used by validation.py, currently 0.999
    "boundary_coverage_threshold",
    # representative point failures from ST_Covers checks
    "boundary_point_failures",
    # SA2 rows whose parent SA4 polygon was missing
    "boundary_missing_parent_sa4",
]

boundary_validation_summary = pd.DataFrame(
    [{key: workflow_summary.get(key) for key in boundary_validation_keys}] # extract validation fields from workflow output
)
display(boundary_validation_summary) # show SA2-to-SA4 validation evidence

,boundary_selected_sa4,boundary_sa2_checked,boundary_sa2_valid,boundary_sa2_invalid,boundary_min_coverage_ratio,boundary_coverage_threshold,boundary_point_failures,boundary_missing_parent_sa4
0,1,27,27,0,1.0,0.999,0,0


## Single-SA4 Database Verification

This query confirms that the rebuilt database contains only `xfan0282`'s selected SA4.


In [42]:
schema = settings.database.schema_name # schema configured in local.yaml, usually data2001

display(pd.read_sql(
    f"""
    SELECT sa4_name, COUNT(*) AS sa2_count
    FROM {schema}.sa2
    GROUP BY sa4_name
    """, # count SA2 rows by SA4 after the selected-SA4 rebuild
    engine, # run the SQL through the PostgreSQL/PostGIS SQLAlchemy engine
))

,sa4_name,sa2_count
0,Sydney - City and Inner South,27


## Task 2 Evidence: API Extraction and Spatial Join

This section checks the POI extraction files and PostGIS spatial assignment outputs.


In [43]:
# Raw API files show that bbox extraction ran for the selected SA4.
# build_sa2_bbox_requests() creates one request per SA2, build_poi_bbox_params() creates ArcGIS envelope params,
# and run_poi_import() paginates NSW POI API responses into raw JSON/JSONL files.
display(load_api_extraction_summary(settings))

# Bbox results are only candidates because rectangles can include points outside the real SA2 polygon.
# Clean POIs are stored in poi_clean, then PostGIS ST_Covers assigns points to SA2 polygons in sa2_poi.
display(load_spatial_join_summary(engine, settings))

,response_dir,response_file_count,features_jsonl,raw_feature_rows,features_file_exists,features_file_size_mb
0,/home/kscii/Codes/data2001-group-assignment/da...,27,/home/kscii/Codes/data2001-group-assignment/da...,3396,True,1.79


,clean_poi,assigned_poi,unassigned_poi,boundary_duplicate_candidates,assignment_rows
0,2203,1718,485,0,1718


## Task 3 Evidence: Score Calculation

This section checks score inputs and stored Task 3 score outputs.


In [44]:
# Score formula in src/data2001/task3_score/scoring.py:
# 1. keep SA2 rows with population >= 100;
# 2. compute mean and population standard deviation of poi_count within xfan0282's selected SA4;
# 3. calculate z_poi = (poi_count - mean_poi_count) / std_poi_count;
# 4. calculate score_raw = sigmoid(z_poi), then score_100 = score_raw * 100.
display(load_score_input_summary(engine, settings))

scores = load_sa2_scores(engine, settings) # scored SA2 rows only, after the population filter
score_map_areas = load_sa2_scores(engine, settings, include_excluded=True) # include excluded SA2 polygons for map background
display(scores.head()) # preview z_poi, score_raw, and score_100 columns
display(build_top_bottom_table(scores, n=settings.charts.top_n)) # show highest and lowest scoring SA2s

,sa2_count,total_poi,mean_poi_count,std_poi_count,min_poi_count,max_poi_count,below_min_population,missing_population
0,27,1718,63.62963,78.506026,4,354,2,0


,sa2_code,sa2_name,sa4_code,sa4_name,population,poi_count,mean_poi_count,std_poi_count,z_poi,score_raw,score_100,is_excluded,exclusion_reason,geometry
0,117011320,Banksmeadow,117,Sydney - City and Inner South,594,4,67.76,80.161477,-0.795395,0.311012,31.101154,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
1,117011321,Botany,117,Sydney - City and Inner South,13254,36,67.76,80.161477,-0.396200,0.402226,40.222560,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
2,117031638,Camperdown - Darlington,117,Sydney - City and Inner South,8452,61,67.76,80.161477,-0.084330,0.478930,47.893004,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
3,117031639,Chippendale,117,Sydney - City and Inner South,8237,11,67.76,80.161477,-0.708071,0.330025,33.002527,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."
4,117031329,Darlinghurst,117,Sydney - City and Inner South,10617,49,67.76,80.161477,-0.234028,0.441759,44.175867,False,None,"{'type': 'MultiPolygon', 'coordinates': [[[[15..."


,rank_group,sa2_code,sa2_name,sa4_name,poi_count,score_100,population
0,top,117021328,Sydenham - Tempe - St Peters,Sydney - City and Inner South,354,97.263629,8395
1,top,117031644,Sydney (North) - Millers Point,Sydney - City and Inner South,304,95.012395,8181
2,top,117031640,Newtown (NSW),Sydney - City and Inner South,92,57.502658,14853
3,top,117031333,Potts Point - Woolloomooloo,Sydney - City and Inner South,72,51.322023,18256
4,top,117011323,Pagewood - Hillsdale - Daceyville,Sydney - City and Inner South,68,50.074849,15398
5,top,117021327,Petersham - Stanmore,Sydney - City and Inner South,65,49.139322,19992
6,top,117031638,Camperdown - Darlington,Sydney - City and Inner South,61,47.893004,8452
7,top,117031336,Surry Hills,Sydney - City and Inner South,61,47.893004,15952
8,top,117031330,Erskineville - Alexandria,Sydney - City and Inner South,59,47.270730,19970
9,top,117031331,Glebe - Forest Lodge,Sydney - City and Inner South,59,47.270730,20628


## Individual Visual Analysis

This section shows the score distribution, spatial pattern, POI structure, and income relationship for the selected SA4.


In [45]:
poi_groups = load_poi_group_counts(engine, settings) # aggregate assigned POIs by POI group from PostGIS
poi_points = load_poi_points(engine, settings, limit=settings.dashboard.poi_limit) # load cleaned POI coordinates for point map
score_income = load_score_income(engine, settings) # join SA2 scores with median income for correlation plot

### Score Distribution

This histogram shows the distribution of well-resourced scores across SA2s in Sydney - City and Inner South. The x-axis is `score_100`, from 0 to 100. The y-axis is the number of SA2s in each score range.

The distribution is uneven rather than flat. Most SA2s sit in the low-to-middle or middle score range, while a small number of SA2s are much higher than the rest. This suggests that POIs are not evenly distributed inside this SA4. A few areas have unusually high POI counts, which pushes their z-scores and sigmoid scores up.


In [46]:
build_score_histogram(scores, nbins=settings.charts.score_histogram_nbins).show() # Plotly histogram of score_100 distribution

### Top and Bottom SA2 Scores

These two bar charts show the highest and lowest scoring SA2s. The x-axis is `score_100`, and the y-axis is the SA2 name. The colour shows the SA4. Since this notebook only analyses one SA4, the colour is mainly kept for a consistent chart format.

The two highest scoring SA2s are Sydenham - Tempe - St Peters and Sydney (North) - Millers Point, with scores of 97.26 and 95.01. They are much higher than the other SA2s; the next highest scored SA2 is far lower at 57.50. After checking the POI point map, these high scores seem to come from very dense records of specific POI types, not from a balanced mix of all resource types.


In [47]:
build_top_sa2_bar(scores, n=settings.charts.top_n).show() # Plotly bar chart for highest scoring SA2s
build_bottom_sa2_bar(scores, n=settings.charts.top_n).show() # Plotly bar chart for lowest scoring SA2s

### SQL Evidence: Top and Bottom Score Values

The SQL below checks the high-score SA2s, `score_100`, POI count, and population used in the explanation above.


In [48]:
score_rank_sql = f"""
SELECT
    s.sa2_name,
    sc.poi_count,
    ROUND(CAST(sc.score_100 AS numeric), 2) AS score_100,
    s.population
FROM {schema}.sa2_score sc
JOIN {schema}.sa2 s
  ON s.sa2_code = sc.sa2_code
WHERE s.sa4_name = %(member_sa4)s
ORDER BY sc.score_100 DESC
LIMIT 5
""" # verify top SA2 scores directly from PostGIS tables

display(pd.read_sql(score_rank_sql, engine, params={"member_sa4": member_sa4})) # parameterised SQL keeps SA4 filter explicit

,sa2_name,poi_count,score_100,population
0,Sydenham - Tempe - St Peters,354,97.26,8395
1,Sydney (North) - Millers Point,304,95.01,8181
2,Newtown (NSW),92,57.50,14853
3,Potts Point - Woolloomooloo,72,51.32,18256
4,Pagewood - Hillsdale - Daceyville,68,50.07,15398


### Score Choropleth Map

This choropleth map shows the spatial distribution of well-resourced scores by SA2. Darker colours mean higher scores. The hover data includes SA2 name, population, POI count, `z_poi`, and score.

The map helps show whether high scores are spatially clustered. Sydenham - Tempe - St Peters and Millers Point appear as clear high-score areas. The POI point map later shows that these high scores are mainly caused by dense records of specific POI types.


In [49]:
build_score_choropleth_map(score_map_areas).show() # Plotly/PostGIS choropleth of score_100 by SA2 polygon

### Population-Adjusted POI Density Map

Each SA2 polygon is coloured by `poi_per_1000`, which means the number of assigned POIs per 1,000 residents. The value is calculated from `poi_count / population * 1000`.

This is an extension beyond the basic Task 3 score. The official score is based on raw POI count after the population threshold, but the density map helps check whether a high score is simply due to large population areas or whether small-population areas have unusually concentrated POIs. It also makes the population filter easier to explain to the marker.


In [50]:
build_poi_density_choropleth_map(score_map_areas).show() # map assigned POIs per 1,000 residents by SA2 polygon


### POI Point Map

This scatter map shows the locations of cleaned POIs. Each point is one POI, and the colour shows the POI group, such as Transport, Recreation, or Community.

A clear Transport line can be seen from Sydenham - Tempe - St Peters towards Newtown and Camperdown - Darlington. The SQL check shows that Sydenham - Tempe - St Peters has 310 Roadside Emergency Telephone POIs, Newtown has 37, and Camperdown - Darlington has 12. This line pattern is mainly caused by roadside emergency telephones, not by a wide mix of public resources.

There is also a regular group of wharf POIs near Millers Point. The SQL check shows that Sydney (North) - Millers Point has 74 Wharf POIs, which is an important reason for its high POI count.

In several south-eastern SA2s, the share of Recreation POIs is clearly higher. For example, Recreation POIs make up 69.1% of Pagewood - Hillsdale - Daceyville, 61.4% of Mascot, and 56.5% of Rosebery - Beaconsfield. Many of these POIs are parks or sports-related locations, so different areas have different POI structures.


In [51]:
build_poi_point_scatter_map(poi_points).show() # point map of cleaned POIs coloured by POI group

### SQL Evidence: POI Point-Map Patterns

The SQL below checks the Roadside Emergency Telephone pattern, the Wharf pattern, and the Recreation POI share in the south-eastern SA2s.


In [52]:
roadside_sql = f"""
SELECT
    s.sa2_name,
    p.poigroup_name,
    p.poitype,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE p.poitype = 'Roadside Emergency Telephone'
GROUP BY s.sa2_name, p.poigroup_name, p.poitype
ORDER BY poi_count DESC, s.sa2_name
LIMIT 10
""" # check whether roadside emergency telephones explain the Sydenham high score

wharf_sql = f"""
SELECT
    s.sa2_name,
    p.poigroup_name,
    p.poitype,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE p.poitype = 'Wharf'
GROUP BY s.sa2_name, p.poigroup_name, p.poitype
ORDER BY poi_count DESC, s.sa2_name
LIMIT 10
""" # check whether wharf POIs explain the Millers Point high score

recreation_share_sql = f"""
WITH counts AS (
    SELECT
        s.sa2_name,
        COUNT(*) AS total_poi,
        COUNT(*) FILTER (WHERE p.poigroup_name = 'Recreation') AS recreation_poi
    FROM {schema}.poi_clean p
    JOIN {schema}.sa2_poi sp
      ON sp.poi_objectid = p.objectid
    JOIN {schema}.sa2 s
      ON s.sa2_code = sp.sa2_code
    WHERE s.sa2_name IN (
        'Pagewood - Hillsdale - Daceyville',
        'Mascot',
        'Rosebery - Beaconsfield',
        'Zetland',
        'Eastlakes',
        'Botany',
        'Waterloo',
        'Banksmeadow'
    )
    GROUP BY s.sa2_name
)
SELECT
    sa2_name,
    total_poi,
    recreation_poi,
    ROUND(CAST(recreation_poi AS numeric) / NULLIF(total_poi, 0) * 100, 1) AS recreation_pct
FROM counts
ORDER BY recreation_pct DESC
""" # calculate recreation share for south-eastern SA2 interpretation

display(pd.read_sql(roadside_sql, engine)) # show Roadside Emergency Telephone counts by SA2
display(pd.read_sql(wharf_sql, engine)) # show Wharf counts by SA2
display(pd.read_sql(recreation_share_sql, engine)) # show Recreation POI share by selected SA2

,sa2_name,poigroup_name,poitype,poi_count
0,Sydenham - Tempe - St Peters,Transport,Roadside Emergency Telephone,310
1,Newtown (NSW),Transport,Roadside Emergency Telephone,37
2,Camperdown - Darlington,Transport,Roadside Emergency Telephone,12
3,Petersham - Stanmore,Transport,Roadside Emergency Telephone,4
4,Darlinghurst,Transport,Roadside Emergency Telephone,2
5,Sydney Airport,Transport,Roadside Emergency Telephone,2
6,Botany,Transport,Roadside Emergency Telephone,1
7,Eastlakes,Transport,Roadside Emergency Telephone,1
8,Mascot,Transport,Roadside Emergency Telephone,1


,sa2_name,poigroup_name,poitype,poi_count
0,Sydney (North) - Millers Point,Transport,Wharf,74
1,Pyrmont,Transport,Wharf,15
2,Potts Point - Woolloomooloo,Transport,Wharf,13
3,Sydney (South) - Haymarket,Transport,Wharf,10
4,Banksmeadow,Transport,Wharf,2
5,Sydney Airport,Transport,Wharf,2
6,Glebe - Forest Lodge,Transport,Wharf,1
7,Port Botany Industrial,Transport,Wharf,1
8,Sydenham - Tempe - St Peters,Transport,Wharf,1


,sa2_name,total_poi,recreation_poi,recreation_pct
0,Pagewood - Hillsdale - Daceyville,68,47,69.1
1,Mascot,44,27,61.4
2,Rosebery - Beaconsfield,23,13,56.5
3,Zetland,11,6,54.5
4,Eastlakes,36,19,52.8
5,Botany,36,16,44.4
6,Waterloo,21,7,33.3
7,Banksmeadow,4,0,0.0


### POI Group Distribution

This chart shows the number of POIs in each POI group for the current SA4. The x-axis is POI count, and the y-axis is POI group.

Transport, Recreation, and Community are the largest POI groups, with 555, 482, and 465 POIs. The high Transport count partly reflects dense Roadside Emergency Telephone and Wharf records, so the score may be sensitive to repeated infrastructure-style POIs. This is important for interpreting the scoring method: the formula is mathematically correct, but raw POI counts can over-reward repeated point features if a POI type has many closely spaced records.


In [53]:
build_poi_group_distribution(poi_groups).show() # Plotly bar chart of assigned POIs by POI group

### SQL Evidence: POI Group Totals

The SQL below checks the Transport, Recreation, and Community POI counts used in the explanation above.


In [54]:
poi_group_sql = f"""
SELECT
    p.poigroup_name,
    COUNT(*) AS poi_count
FROM {schema}.poi_clean p
JOIN {schema}.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN {schema}.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE s.sa4_name = %(member_sa4)s
GROUP BY p.poigroup_name
ORDER BY poi_count DESC
""" # verify POI group totals directly from PostGIS

print(poi_group_sql) # print SQL so the grouping logic is visible in the notebook
display(pd.read_sql(poi_group_sql, engine, params={"member_sa4": member_sa4})) # show group counts for xfan0282's SA4


SELECT
    p.poigroup_name,
    COUNT(*) AS poi_count
FROM data2001.poi_clean p
JOIN data2001.sa2_poi sp
  ON sp.poi_objectid = p.objectid
JOIN data2001.sa2 s
  ON s.sa2_code = sp.sa2_code
WHERE s.sa4_name = %(member_sa4)s
GROUP BY p.poigroup_name
ORDER BY poi_count DESC



,poigroup_name,poi_count
0,Transport,555
1,Recreation,482
2,Community,465
3,Education,109
4,Place,77
5,Landform,14
6,Hydrography,10
7,Utility,6


### Score and Median Income

This scatter plot shows the relationship between median income and well-resourced score. The x-axis is `median_income_2022_23`, the y-axis is `score_100`, the point size shows POI count, and the colour shows SA4.

The plot does not show a clear increase in score as income increases. Pearson correlation is 0.0814 with a p-value of 0.7054. Spearman correlation is 0.3065 with a p-value of 0.1452. Both p-values are above 0.05, so the relationship between median income and score is not statistically significant in this SA4 sample.

Sydenham - Tempe - St Peters and Millers Point have much higher scores than other SA2s at ordinary or similar income levels. Based on the POI type checks above, this is more likely caused by dense Roadside Emergency Telephone and Wharf records than by income itself.


In [55]:
build_score_income_scatter(score_income).show() # Plotly scatter of score_100 against median_income_2022_23

### SQL Evidence: Income-Score Correlation

The SQL below checks the Pearson/Spearman correlation results and the high-score SA2s in the income-score scatter plot.


In [56]:
correlation_sql = f"""
SELECT
    method,
    ROUND(CAST(statistic AS numeric), 4) AS statistic,
    ROUND(CAST(p_value AS numeric), 4) AS p_value,
    n,
    alpha,
    is_significant
FROM {schema}.score_income_correlation
ORDER BY method
""" # read Pearson and Spearman correlation tests produced by compute_score

top_income_sql = f"""
SELECT
    s.sa2_name,
    sc.poi_count,
    ROUND(CAST(sc.score_100 AS numeric), 2) AS score_100,
    i.median_income_2022_23,
    i.income_earners_2022_23
FROM {schema}.sa2_score sc
JOIN {schema}.sa2 s
  ON s.sa2_code = sc.sa2_code
JOIN {schema}.sa2_income i
  ON i.sa2_code = sc.sa2_code
WHERE s.sa4_name = %(member_sa4)s
ORDER BY sc.score_100 DESC
LIMIT 5
""" # compare high-score SA2s with median income values

display(pd.read_sql(correlation_sql, engine)) # show correlation statistic, p-value, sample size, and significance flag
display(pd.read_sql(top_income_sql, engine, params={"member_sa4": member_sa4})) # show income context for top scoring SA2s

,method,statistic,p_value,n,alpha,is_significant
0,pearson,0.0814,0.7054,24,0.05,False
1,spearman,0.3065,0.1452,24,0.05,False


,sa2_name,poi_count,score_100,median_income_2022_23,income_earners_2022_23
0,Sydenham - Tempe - St Peters,354,97.26,76736,6389
1,Sydney (North) - Millers Point,304,95.01,52730,11869
2,Newtown (NSW),92,57.50,74226,12098
3,Potts Point - Woolloomooloo,72,51.32,74599,15995
4,Pagewood - Hillsdale - Daceyville,68,50.07,60099,10268


## Correlation Summary

This final table labels the latest Pearson and Spearman tests as statistically significant or not significant.


In [57]:
display(load_correlation_summary(engine, settings)) # load latest correlation rows and add plain-language significance labels

,method,statistic,p_value,n,alpha,is_significant,created_at,interpretation
0,pearson,0.081378,0.705419,24,0.05,False,2026-05-20 13:01:38.015896+00:00,not statistically significant
1,spearman,0.306487,0.145215,24,0.05,False,2026-05-20 13:01:38.015896+00:00,not statistically significant


## Final Key Findings

- The selected SA4 for `xfan0282` is `Sydney - City and Inner South`. This personal workflow rebuilds the database for this SA4 only, so the notebook is individual Task 2/3 evidence rather than a full Greater Sydney analysis.
- The Task 2 workflow meets the main extraction requirements: it has one bbox POI request per SA2 (`sa2_bbox_requests = 27`), stores raw API features, loads cleaned POIs into PostgreSQL/PostGIS, and assigns POIs to real SA2 polygons with `ST_Covers`.
- The boundary validation supports the SA4 scope: 27 SA2 polygons were checked, 27 were valid, and 0 were invalid against the selected SA4.
- The Task 3 scoring workflow starts with 27 SA2 areas and excludes 2 areas because their population is below 100, leaving 25 scored SA2s.
- The scores vary strongly within this SA4. The two highest scoring SA2s are `Sydenham - Tempe - St Peters` and `Sydney (North) - Millers Point`, with scores of 97.26 and 95.01, while most other SA2s are much lower.
- The main reason for the strongest variation is POI composition, not income. `Sydenham - Tempe - St Peters` is strongly affected by Roadside Emergency Telephone POIs, while `Sydney (North) - Millers Point` is strongly affected by Wharf POIs.
- The score-income relationship is not statistically significant in this SA4 sample. Pearson correlation is 0.0814 with p-value 0.7054, and Spearman correlation is 0.3065 with p-value 0.1452. This suggests that median income alone does not explain the score pattern here.
- The scoring method is suitable here because it applies the z-score plus sigmoid formula and the population filter. The main caveat is that equal-weight raw POI counts can overemphasise repeated infrastructure-style POIs, so the extension analysis is needed to interpret the result responsibly.
